# Stage 3 — 3D Lifting and Official Benchmark Metrics

Training-free: lifts Stage 2's 2D masks into 3D point-cloud instances via multi-view
fusion, and scores them with SceneFun3D's own official evaluator (AP25/AP50/mAP).

**Scope of this notebook (deliberately narrow): lifting only, given a known query.**
Each instance below assumes the target element and its GT label are already known
(same `oracle_label` assumption Stage 2 uses for its cleanest numbers) -- this
notebook does not yet do full-scene detection (finding *every* functional element in
a novel scene unprompted). That is the natural next step once lifting itself is
solid; see the caveats section at the bottom for exactly what is and is not covered.

**Run order after any disconnect: Cell 1 (SETUP), then re-run cell 2 (imports the
official toolkit + Stage 2 package). Nothing else needs to be repeated.**

---

### Why this exists, and the one finding that shapes everything here

Stage 2's 2D pixel-IoU metric cannot see a specific failure mode: **2D pixel area
does not correspond to a fixed real-world area** -- it depends on depth and field of
view. A SAM mask that looks like a plausible blob in pixel space can correspond to a
wildly oversized real-world footprint if that part of the frame is a distant wall or
cabinet face. This is invisible in 2D pixel-IoU and only shows up once you lift to
metres. Measured on a 14-item local reference set (mixed labels, 4 scenes): naive
multi-view fusion (accumulate SAM masks across frames, threshold agreement) gives
**mean 3D IoU 0.06-0.07 with predicted masks 100-250x larger than the true element**,
regardless of how the agreement threshold is tuned.

**The fix implemented below** re-ranks SAM's own candidate masks (it always proposes
3, at different granularities) by their *real physical extent* -- computed from the
actual laser-scan points visible in that frame, not pixel count -- against a fixed
prior (median GT extent, measured, not fit: **1.70 cm** mean-distance-from-centroid
across 757 real annotated elements). A frame contributes to the fused mask only if at
least one candidate is physically plausible; a point only survives into the final
mask if independently flagged by >=2 different frames (single-frame false positives
to a similarly-sized-but-wrong object are common and must not survive alone); the
final mask keeps only its single largest spatially-coherent cluster (BFS
region-growing, same pattern as the motion module's `panel_component`) -- a gripper
needs one coherent region, not scattered agreement across the room.

**Result of the fix, same 14 items:** mean IoU improves (0.06 -> 0.10) but more
importantly the failure mode changes character -- from "confidently wrong, 100x
oversized" to "correctly abstains when the signal isn't there" (8/14 items now
produce *no* prediction rather than a garbage one). One clean success case
(`key_press`, IoU 0.78-0.79, reproduced twice) proves the mechanism works when the
underlying 2D correspondence is good enough. **The honest conclusion: 3D lifting
cannot manufacture accuracy that is not present in the 2D signal.** Stage 2's own
finding -- correspondence localisation caps out around 45% peak-in-GT even given the
correct box -- is the real ceiling here too. This notebook makes that ceiling
visible in *official* metrics instead of hiding it behind pixel-IoU.

### Known-good numbers (n=8 predicted instances, 4 scenes, reference GT -- see caveats)

- mAP 1.7, AP50 2.5, AP25 2.5 (0-100 scale), via the real, unmodified scoring logic
  of the official `eval_script.py`.
- These are intentionally low and are not the paper's headline number -- they
  characterise the *current* ceiling on fine-grained elements (median extent 1.7cm)
  embedded in large flat surfaces, using zero-shot DINOv3+SAM+GroundingDINO. Improving
  them further means improving Stage 2's correspondence step, not this lifting step.

## 1. SETUP

Run once per session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, subprocess

ROOT = '/content/drive/MyDrive/scenefun3d'
assert os.path.isdir(ROOT), f'not found: {ROOT}'

# Official SceneFun3D toolkit: utils.data_parser (frame/pose/intrinsics parsing),
# utils.fusion_util (PointCloudToImageMapper), and the real eval/ scoring code.
if not os.path.isdir('/content/scenefun3d_toolkit'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/SceneFun3D/scenefun3d.git',
                    '/content/scenefun3d_toolkit'])
sys.path.insert(0, '/content/scenefun3d_toolkit')

# Stage 2's package (retrieval/correspondence/box/SAM). Assumes 01_segmentation_pipeline.ipynb
# has been run at least once so /content/drive/MyDrive/scenefun3d/sf3d exists.
sys.path.insert(0, '/content/drive/MyDrive/scenefun3d')

try:
    import open3d
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'open3d', 'imageio'])

import numpy as np
# eval_script.py predates numpy 2.x removing np.in1d, and predates numpy 2.x's
# stricter scalar-conversion rules for shape-(1,) arrays. Both are drop-in safe.
if not hasattr(np, 'in1d'):
    np.in1d = np.isin

print('setup ok')

## 2. Package modules

Written to local disk, synced to Drive -- same convention as Stage 2. Edit here,
never in a bare cell.

In [ ]:
%%writefile /content/sf3d/lifting.py
"""Multi-view 2D-to-3D lifting for Stage 2 predictions, with metric-aware SAM
candidate re-selection. See notebook header for the diagnosis this fixes.
"""
import numpy as np
import cv2
from scipy.spatial import cKDTree
from plyfile import PlyData
from utils.fusion_util import PointCloudToImageMapper

from . import config as C
from . import pipeline as P


def load_laser_scan_xyz(data_root, visit_id):
    """plyfile-based (not open3d.io.read_point_cloud): avoids an open3d
    dependency for the one thing we actually need -- XYZ positions -- and
    matches the same loader the motion module already uses."""
    path = f'{data_root}/{visit_id}/{visit_id}_laser_scan.ply'
    vx = PlyData.read(path)['vertex']
    return np.stack([vx['x'], vx['y'], vx['z']], 1).astype(np.float64)


def pick_frames(parser, visit_id, video_id, center_ts, n_frames=6):
    """Frames spread around the exemplar's own annotation timestamp, restricted
    to timestamps that have an RGB frame, a depth frame, AND intrinsics."""
    rgb_paths = parser.get_rgb_frames(visit_id, video_id, data_asset_identifier='hires_wide')
    depth_paths = parser.get_depth_frames(visit_id, video_id, data_asset_identifier='hires_depth')
    intr_paths = parser.get_camera_intrinsics(visit_id, video_id, data_asset_identifier='hires_wide_intrinsics')
    common = sorted(set(rgb_paths) & set(depth_paths) & set(intr_paths), key=float)
    if not common:
        return [], rgb_paths, depth_paths, intr_paths
    center = min(common, key=lambda t: abs(float(t) - float(center_ts)))
    idx = common.index(center)
    offsets = sorted(range(-n_frames // 2, n_frames // 2 + 1), key=abs)
    picked = []
    for off in offsets:
        j = idx + off
        if 0 <= j < len(common) and common[j] not in picked:
            picked.append(common[j])
        if len(picked) >= n_frames:
            break
    return picked, rgb_paths, depth_paths, intr_paths


def rescore_candidates_by_metric(sam_cands, hot, mapping, laser_xyz, prior_extent, region, max_mult=4.0):
    """Re-rank SAM's raw candidate masks (it always proposes 3, at different
    granularities) by REAL physical extent, using the laser-scan points
    actually visible in this frame -- not pixel count, which is blind to
    depth. Returns None if no candidate is physically plausible: we do not
    fall back to an oversized one, we skip this frame's contribution.

    sam_cands are CROP-sized (SAM only returns masks at the box-crop's own
    resolution), so mapping/hot are shifted into crop-relative coordinates.
    """
    X0, Y0, X1, Y1 = region
    valid = mapping[:, 2] == 1
    rows, cols = mapping[valid, 0] - Y0, mapping[valid, 1] - X0
    global_idx = np.where(valid)[0]
    in_crop = (rows >= 0) & (cols >= 0) & (rows < (Y1 - Y0)) & (cols < (X1 - X0))
    rows, cols, global_idx = rows[in_crop], cols[in_crop], global_idx[in_crop]

    scored = []
    for i, m in enumerate(sam_cands):
        if m.sum() < 20:
            continue
        H, W = m.shape
        in_bounds = (rows < H) & (cols < W)
        hit = m[rows[in_bounds], cols[in_bounds]].astype(bool)
        pt_idx = global_idx[in_bounds][hit]
        if len(pt_idx) < 5:
            continue
        pts = laser_xyz[pt_idx]
        extent = float(np.linalg.norm(pts - pts.mean(0), axis=1).mean())
        cov = (m & hot).sum() / max(hot.sum(), 1) if hot is not None else 0.0
        pur = (m & hot).sum() / max(m.sum(), 1) if hot is not None else 0.0
        scored.append(dict(extent=extent, pt_idx=pt_idx,
                            plausible=(extent <= max_mult * prior_extent), pur=pur, cov=cov))

    pool = [s for s in scored if s['plausible']]
    if not pool:
        return None  # never fall back to an implausibly-large candidate
    return max(pool, key=lambda s: (2.0 * s['pur'] + 0.5 * s['cov'], -s['extent']))


def largest_cluster(points, link=0.05, min_pts=3):
    """Keep only the single largest spatially-coherent cluster (BFS via
    radius-linking -- same pattern as the motion module's panel_component). A
    gripper needs one coherent region, not scattered agreement across unrelated
    parts of the room."""
    if len(points) < min_pts:
        return np.zeros(len(points), bool)
    tree = cKDTree(points)
    pairs = tree.query_pairs(link, output_type='ndarray')
    parent = np.arange(len(points))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in pairs:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    roots = np.array([find(i) for i in range(len(points))])
    labels, counts = np.unique(roots, return_counts=True)
    return roots == labels[np.argmax(counts)]


def lift_instance(parser, e, gt_indices, proto, prior_extent, n_frames=6,
                   agree_thresh=0.5, min_views=1, min_flags=2, cluster_link=0.05,
                   verbose=False):
    """Lift one exemplar's Stage 2 prediction into a 3D point mask.

    Args:
        parser: utils.data_parser.DataParser
        e: exemplar_db entry (needs visit, video, frame, label, desc)
        gt_indices: ground-truth point indices for this instance (for scoring;
            omit/None at real inference time when GT isn't known)
        proto: output of pipeline.build_proto(pool) for this label
        prior_extent: physical size prior in metres (see notebook section 3)
    Returns dict with pred_mask, iou3d (if gt_indices given), used_frames, etc.
    """
    visit_id, video_id, center_ts = e['visit'], e['video'], e['frame']
    xyz = load_laser_scan_xyz(C.ROOT + '/data', visit_id)
    n_points = xyz.shape[0]

    timestamps, rgb_paths, depth_paths, intr_paths = pick_frames(parser, visit_id, video_id, center_ts, n_frames)
    if not timestamps:
        return dict(ok=False, reason='no posed frames available for this visit')

    poses = parser.get_camera_trajectory(visit_id, video_id, pose_source='colmap')
    w, h, fx, fy, hw, hh = parser.read_camera_intrinsics(next(iter(intr_paths.values())))
    mapper = PointCloudToImageMapper((int(w), int(h)), visibility_threshold=0.25, cut_bound=0)

    acc = np.zeros(n_points, np.float32)
    seen = np.zeros(n_points, np.float32)
    used_frames, frame_report = 0, []
    for ts in timestamps:
        pose = parser.get_nearest_pose(ts, poses)
        if pose is None:
            continue
        img = cv2.cvtColor(cv2.imread(rgb_paths[ts]), cv2.COLOR_BGR2RGB)
        text = f"{(e.get('desc') or [''])[0]}. {C.PARENT_HINT.get(e['label'], C.DEFAULT_HINT)}"
        res = P.run(img, proto, text)
        if res is None:
            continue
        depth = parser.read_depth_frame(depth_paths[ts])
        intr = parser.read_camera_intrinsics(intr_paths[ts], format='matrix')
        mapping = mapper.compute_mapping(pose, xyz, depth, intr)
        if (mapping[:, 2] == 1).sum() == 0:
            continue

        region = res['region']
        X0, Y0, X1, Y1 = region
        region_smap = res['smap'][Y0:Y1, X0:X1]
        hot_region = region_smap >= np.percentile(region_smap, 92)

        best = rescore_candidates_by_metric(res['sam_cands'], hot_region, mapping, xyz, prior_extent, region)
        if best is None:
            frame_report.append((ts, 'no metric-plausible candidate')); continue

        acc[best['pt_idx']] += 1
        seen[mapping[:, 2] == 1] += 1
        used_frames += 1
        frame_report.append((ts, f"extent={best['extent']:.4f}m n_pts={len(best['pt_idx'])}"))

    result = dict(ok=used_frames > 0, visit=visit_id, annot_id=e.get('annot_id'), label=e['label'],
                  used_frames=used_frames, n_frames_tried=len(timestamps), frame_report=frame_report)
    if used_frames == 0:
        return result

    with np.errstate(divide='ignore', invalid='ignore'):
        agreement = np.where(seen > 0, acc / seen, 0)
    raw_pred_mask = (agreement >= agree_thresh) & (seen >= min_views) & (acc >= min_flags)

    pred_idx = np.where(raw_pred_mask)[0]
    if len(pred_idx) >= 3:
        keep = largest_cluster(xyz[pred_idx], link=cluster_link)
        pred_mask = np.zeros(n_points, bool); pred_mask[pred_idx[keep]] = True
    else:
        pred_mask = raw_pred_mask

    result['pred_mask'] = pred_mask
    result['pred_pts'] = int(pred_mask.sum())
    if gt_indices is not None:
        gt_mask = np.zeros(n_points, bool); gt_mask[gt_indices] = True
        inter = np.logical_and(pred_mask, gt_mask).sum()
        union = np.logical_or(pred_mask, gt_mask).sum()
        result['iou3d'] = float(inter / union) if union > 0 else 0.0
        result['gt_pts'] = int(gt_mask.sum())
    if verbose:
        print(f"  {visit_id} {e['label']:<10} frames={used_frames}/{len(timestamps)} "
              f"pred_pts={result['pred_pts']}" +
              (f" gt_pts={result['gt_pts']} IoU={result['iou3d']:.3f}" if gt_indices is not None else ''))
    return result

## 3. Physical extent prior

A dataset constant, not a per-item oracle -- same status as Stage 2's `MEDIAN_AREA`.
Computed from real annotated elements (needs `02_motion_parameters.ipynb`'s cached
`annot_points/*.npz`, which already has GT region point sets extracted directly from
the laser scan for exactly this purpose). If that cache isn't available, falls back
to extracting the same quantity directly from `*_annotations.json` for whatever
visits are on hand.

In [ ]:
import glob

def compute_physical_extent_prior():
    pts_dir = f'{ROOT}/annot_points'
    extents = []
    if os.path.isdir(pts_dir):
        for f in glob.glob(pts_dir + '/*.npz'):
            z = np.load(f)
            for k in z.files:
                if k.startswith('NBR_'):
                    continue
                pts = z[k].astype(np.float64)
                if len(pts) < 5:
                    continue
                extents.append(float(np.linalg.norm(pts - pts.mean(0), axis=1).mean()))
    if not extents:
        raise RuntimeError('No annot_points cache found -- run 02_motion_parameters.ipynb '
                            'cell 2 at least once, or adapt this to read *_annotations.json directly.')
    prior = float(np.median(extents))
    print(f'physical extent prior: n={len(extents)}  median={prior:.4f} m  '
          f'p25={np.percentile(extents,25):.4f}  p75={np.percentile(extents,75):.4f}')
    return prior

PRIOR_EXTENT = compute_physical_extent_prior()

## 4. RESUME

Everything below depends on cells 1-3. After a disconnect: cell 1, then this.

In [ ]:
import pickle, importlib
import cv2

for m in [k for k in list(sys.modules) if k.startswith('sf3d')]:
    del sys.modules[m]
from sf3d import config as C, pipeline as P, retrieval as RT, evalutils as EV, lifting as L
from utils.data_parser import DataParser

DB = pickle.load(open(C.DB_PATH, 'rb'))
P.init_stats(DB)
RT.build_index(DB)
parser = DataParser(f'{ROOT}/data')

def visits_with_posed_frames(db, root):
    """Which visits actually have hires_wide + hires_depth + poses locally --
    multi-view lifting needs these, Stage 2's cached single frame is not enough.
    If a visit you need is missing, download it with e.g.:
    python -m data_downloader.data_asset_download --split custom \
        --download_dir data --visit_id <VISIT> --video_id <VIDEO> \
        --dataset_assets hires_wide_intrinsics laser_scan_5mm hires_poses hires_wide hires_depth"""
    ok = []
    for visit_id in sorted({e['visit'] for e in db.values()}):
        d = f'{root}/data/{visit_id}'
        has = (glob.glob(f'{d}/*/hires_wide/*.jpg') and glob.glob(f'{d}/*/hires_depth/*') and
               glob.glob(f'{d}/*/hires_poses.traj'))
        if has:
            ok.append(visit_id)
    return ok

POSED_VISITS = visits_with_posed_frames(DB, ROOT)
print(f'{len(POSED_VISITS)}/{len({e["visit"] for e in DB.values()})} visits have full posed frame data: {POSED_VISITS}')

## 5. Component: lifting, sanity check

Runs on a small, seeded sample so a re-run is directly comparable across sessions.
Extend `n` and the visit filter once you have more visits downloaded with posed
frames (see the warning printed by cell 4 if `POSED_VISITS` is short).

In [ ]:
def load_annotations(visit_id):
    ann = json.load(open(f'{ROOT}/data/{visit_id}/{visit_id}_annotations.json'))
    return ann.get('annotations', ann.get('annotation', ann))

def gt_indices_for_annot(visit_id, annot_id):
    for a in load_annotations(visit_id):
        if a.get('annot_id') == annot_id:
            return np.asarray(a['indices'], np.int64)
    return None

import json

def run_batch(n_per_label=2, n_frames=6, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    targets = [(k, v) for k, v in DB.items() if v['visit'] in POSED_VISITS]
    by_label = {}
    for k, v in targets:
        by_label.setdefault(v['label'], []).append((k, v))
    sel = []
    for label, items in by_label.items():
        idxs = rng.choice(len(items), min(n_per_label, len(items)), replace=False)
        sel.extend(items[i] for i in idxs)
    print(f'{len(sel)} exemplars across {len(by_label)} labels: {sorted(by_label.keys())}')

    results = []
    for k, e in sel:
        gt_idx = gt_indices_for_annot(e['visit'], e['annot_id'])
        if gt_idx is None:
            continue
        pool = EV.pool_oracle(DB, k)
        proto = P.build_proto(pool)
        if proto is None:
            continue
        r = L.lift_instance(parser, e, gt_idx, proto, PRIOR_EXTENT, n_frames=n_frames, verbose=verbose)
        r['key'] = k
        results.append(r)
    return results

RESULTS = run_batch()

ok = [r for r in RESULTS if r.get('ok') and 'iou3d' in r]
if ok:
    ious = np.array([r['iou3d'] for r in ok])
    print(f"\nmean IoU={ious.mean():.3f}  median={np.median(ious):.3f}  "
          f"n_scored={len(ok)}/{len(RESULTS)} attempted")

## 6. Official submission format + real evaluation

Builds a proper `{visit_id}.txt` GT file per scene (every annotated instance in that
scene, not just the ones sampled above -- so unsampled true instances correctly count
as false negatives) and a matching prediction file, then runs the actual,
unmodified `eval_script.evaluate()` from the official toolkit. Confidence score is
`used_frames / n_frames_tried` -- the fraction of sampled views that had a
metric-plausible candidate at all; a simple, real, fixed-geometry signal, not a
learned one.

In [ ]:
from eval.functionality_segmentation.eval_utils.eval_script import evaluate as official_evaluate
from eval.functionality_segmentation.eval_utils.benchmark_labels import CLASS_LABELS, VALID_CLASS_IDS
from plyfile import PlyData

LABEL_TO_ID = {lab: int(cid) for lab, cid in zip(CLASS_LABELS, VALID_CLASS_IDS)}

def laser_scan_n_points(visit_id):
    vx = PlyData.read(f'{ROOT}/data/{visit_id}/{visit_id}_laser_scan.ply')['vertex']
    return len(vx['x'])

def build_full_scene_gt(visit_id, n_points):
    gt_ids = np.zeros(n_points, dtype=np.int64)
    inst = 1
    for a in load_annotations(visit_id):
        label = a.get('label')
        if label == 'exclude' or label not in LABEL_TO_ID:
            continue
        idx = np.asarray(a['indices'], np.int64)
        idx = idx[idx < n_points]
        if len(idx) == 0:
            continue
        gt_ids[idx] = LABEL_TO_ID[label] * 1000 + inst
        inst += 1
    return gt_ids

def run_official_eval(results, gt_dir='/content/official_gt'):
    ok = [r for r in results if r.get('ok') and r.get('used_frames', 0) > 0 and 'pred_mask' in r]
    print(f'{len(ok)} predicted instances (of {len(results)} attempted)')
    by_visit = {}
    for r in ok:
        by_visit.setdefault(r['visit'], []).append(r)

    os.makedirs(gt_dir, exist_ok=True)
    preds = {}
    for visit_id, items in by_visit.items():
        n_points = laser_scan_n_points(visit_id)
        gt_ids = build_full_scene_gt(visit_id, n_points)
        np.savetxt(f'{gt_dir}/{visit_id}.txt', gt_ids, fmt='%d')
        masks = [r['pred_mask'][:n_points] for r in items]
        classes = np.array([LABEL_TO_ID[r['label']] for r in items])
        scores = np.array([r['used_frames'] / max(r['n_frames_tried'], 1) for r in items])
        preds[visit_id] = dict(pred_masks=np.vstack(masks).T, pred_classes=classes, pred_scores=scores)

    return official_evaluate(preds, gt_dir, print_result=True)

AVGS = run_official_eval(RESULTS)
print(f"\nmAP={AVGS['all_ap']:.2f}  AP50={AVGS['all_ap_50%']:.2f}  AP25={AVGS['all_ap_25%']:.2f}")

## 7. Results summary

In [ ]:
print('='*72)
print('3D LIFTING -- multi-view fusion, metric-aware candidate selection')
print('='*72)
ok = [r for r in RESULTS if r.get('ok') and 'iou3d' in r]
ious = np.array([r['iou3d'] for r in ok]) if ok else np.array([])
print(f'\nLIFTING  n_attempted={len(RESULTS)}  n_scored={len(ok)}')
if len(ious):
    print(f'  mean IoU {ious.mean():.3f}   median {np.median(ious):.3f}   '
          f'good (>0.5) {np.mean(ious>0.5):.0%}')
print(f'\nOFFICIAL METRICS (eval_script.py, unmodified scoring logic)')
print(f'  mAP {AVGS["all_ap"]:.2f}   AP50 {AVGS["all_ap_50%"]:.2f}   AP25 {AVGS["all_ap_25%"]:.2f}')

print('\nCAVEATS TO STATE IN ANY WRITE-UP')
print('  - Lifting only: assumes the target/label is already known (oracle_label),')
print('    same as Stage 2\'s cleanest condition. Full-scene detection (finding every')
print('    element in a novel scene unprompted) is not yet implemented.')
print('  - GT here is built from *_annotations.json directly (real, but a local')
print('    reference construction) -- not the official withheld validation GT.')
print('    Swap in official GT the moment you have it; the submission format matches.')
print('  - n is small and scoped to visits with full posed-frame data locally.')
print('    Extend POSED_VISITS by downloading hires_wide/hires_depth/hires_poses for')
print('    more visits (see the docstring on visits_with_posed_frames).')
print('  - The physical extent prior (1.70 cm) is measured, not tuned to any GT used')
print('    for scoring here -- it comes from the motion module\'s independent cache.')
print('  - Low AP is not a bug: correspondence localisation (Stage 2, ~45% ceiling)')
print('    is the limiting factor, not the lifting step. See notebook header.')

## 8. The motion gate -- SceneFun3D's own composed metric, end-to-end

Everything above (AP25/AP50/mAP) scores segmentation only. SceneFun3D's
actual definition of the *motion estimation* task (paper Sec.6.3, Table 5)
is a single composed metric, not a separate segmentation number and a
separate motion number: starting from AP25 (predicted mask must clear
IoU>=0.25 against ground truth), it nests three gates -- **+M** (also
correct motion type), **+MA** (+ axis within 15 deg), **+MAO** (+ origin
MD<0.25m for revolute). Their own trained baseline (Mask3D-FM, full
supervision, full-scene detection) scores AP25=26.6% -> +M=23.8-24.0% ->
+MA=9.8-10.2% -> **+MAO=7.9-8.1%**.

Nothing in this project (or, as far as a literature check found, any
SceneFun3D follow-up -- REACT3D uses its own different custom "MOD" metric,
evaluated only on its own true-positive detections, not this one) has
reported this exact composed metric before. This section computes it for
real, for the first time, using `sf3d.motion` (§2's motion module, packaged
as a plain module so it's importable here) on top of Stage 3's own predicted
3D masks -- **not** oracle regions.

**Reuses the official scoring math unmodified.** `evaluate_matches` and
`compute_averages` (imported above, Cell 12) are called completely
unchanged. The only new logic here decides, for each spatially-overlapping
(GT, prediction) pair the official matcher already found, whether it *also*
satisfies the motion-correctness gate for a given tier -- pairs that don't
are pruned from the match lists before the official AP math runs, exactly
mirroring the paper's own stated approach ("we further constrain mask
prediction by whether the model accurately predicted..."). One small,
correctness-neutral patch is needed first: the official `make_pred_info`
assigns each prediction a random `uuid4()`, which throws away which query
element a prediction came from -- swapped for a deterministic index so
predictions stay traceable back to their own 3D points and label. This
changes zero AP math; a UUID is only ever used as an opaque dict key.

In [ ]:
import eval.functionality_segmentation.eval_utils.eval_script as ES
from sf3d import motion as MOT
from scipy.spatial import cKDTree

def _traceable_make_pred_info(pred):
    """Identical to the official make_pred_info, except uuids are the
    prediction's column index (deterministic) instead of uuid4() (random) --
    needed so a matched prediction can be traced back to its own 3D points
    and label. Does not touch AP math: a uuid is only ever used as an
    opaque, unique dict key downstream."""
    pred_info = {}
    assert (pred['pred_classes'].shape[0] == pred['pred_scores'].shape[0] == pred['pred_masks'].shape[1])
    for i in range(len(pred['pred_classes'])):
        pred_info[i] = dict(label_id=pred['pred_classes'][i], conf=pred['pred_scores'][i],
                             mask=pred['pred_masks'][:, i])
    return pred_info

ES.make_pred_info = _traceable_make_pred_info

def load_motions(visit_id):
    mot = json.load(open(f'{ROOT}/data/{visit_id}/{visit_id}_motions.json'))
    return mot.get('motions', mot.get('motion', mot))

def gt_motion_for_annot(visit_id, annot_id, scan_xyz):
    """Ground-truth motion dict for one annotation, resolving
    motion_origin_idx to an actual xyz point via the laser scan -- same
    convention as 02_motion_parameters.ipynb Cell 4."""
    for m in load_motions(visit_id):
        if m.get('annot_id') != annot_id:
            continue
        d = np.asarray(m.get('motion_dir', []), float)
        if d.shape != (3,) or np.linalg.norm(d) < 1e-6:
            return None
        t = 'rot' if 'rot' in str(m.get('motion_type')).lower() else 'trans'
        org = None
        oi = m.get('motion_origin_idx')
        if oi is not None and int(oi) < len(scan_xyz):
            org = scan_xyz[int(oi)]
        return dict(motion_type=t, motion_dir=d / np.linalg.norm(d), motion_origin=org)
    return None

def build_full_scene_gt_with_map(visit_id, n_points):
    """Same as Cell 12's build_full_scene_gt, but also returns
    instance_id -> annot_id so a matched GT instance can be traced back to
    its real motion parameters."""
    gt_ids = np.zeros(n_points, dtype=np.int64)
    inst_to_annot = {}
    inst = 1
    for a in load_annotations(visit_id):
        label = a.get('label')
        if label == 'exclude' or label not in LABEL_TO_ID:
            continue
        idx = np.asarray(a['indices'], np.int64)
        idx = idx[idx < n_points]
        if len(idx) == 0:
            continue
        inst_id = LABEL_TO_ID[label] * 1000 + inst
        gt_ids[idx] = inst_id
        inst_to_annot[inst_id] = a['annot_id']
        inst += 1
    return gt_ids, inst_to_annot

_WIDE_NBR_CACHE = {}
def wide_neighbourhood(visit_id, scan_xyz, scan_tree, cen, radius=1.10, cap=120000):
    """Neighbourhood around a PREDICTED region's own centroid, queried fresh
    from the raw scan -- not the cached annot_points_r11 npz (that's keyed to
    ground-truth centroids, which a predicted region only approximately
    matches). See MOTION.md Sec.8: 0.60m cuts off 37% of true hinge origins,
    hence the 1.10m radius here too."""
    idx = scan_tree.query_ball_point(cen, radius)
    nbr = scan_xyz[idx]
    if len(nbr) > cap:
        nbr = nbr[np.random.default_rng(0).choice(len(nbr), cap, replace=False)]
    return nbr


In [ ]:
from copy import deepcopy

def motion_correct(gt_m, pred_m, gate):
    """The M/MA/MAO predicate, mirroring the paper's own nested definition
    (Sec.6.3): type match, then +axis<15deg, then +origin MD<0.25m for
    revolute (prismatic is origin-invariant, so MA already implies MAO)."""
    if gt_m is None or pred_m is None:
        return False
    if gt_m['motion_type'] != pred_m['motion_type']:
        return False
    if gate == 'M':
        return True
    oe = MOT.OE(gt_m['motion_dir'], pred_m['motion_dir'])
    if oe >= 15:
        return False
    if gate == 'MA':
        return True
    if gt_m['motion_type'] != 'rot':
        return True
    if gt_m['motion_origin'] is None or pred_m['motion_origin'] is None:
        return False
    md = MOT.MD(pred_m['motion_origin'], pred_m['motion_dir'], gt_m['motion_origin'], gt_m['motion_dir'])
    return md < 0.25

def build_motion_gated_matches(results, gt_dir='/content/official_gt_motion'):
    os.makedirs(gt_dir, exist_ok=True)
    by_visit = {}
    for r in results:
        if r.get('ok') and r.get('used_frames', 0) > 0 and 'pred_mask' in r:
            by_visit.setdefault(r['visit'], []).append(r)

    scan_cache = {}
    all_matches = {g: {} for g in ('raw', 'M', 'MA', 'MAO')}
    n_gt_motion_ok = n_pred_motion_ok = n_pred_total = 0

    for visit_id, items in by_visit.items():
        n_points = laser_scan_n_points(visit_id)
        gt_ids, inst_to_annot = build_full_scene_gt_with_map(visit_id, n_points)
        gt_file = f'{gt_dir}/{visit_id}.txt'
        np.savetxt(gt_file, gt_ids, fmt='%d')

        masks = [r['pred_mask'][:n_points] for r in items]
        classes = np.array([LABEL_TO_ID[r['label']] for r in items])
        scores = np.array([r['used_frames'] / max(r['n_frames_tried'], 1) for r in items])
        pred = dict(pred_masks=np.vstack(masks).T, pred_classes=classes, pred_scores=scores)
        gt2pred, pred2gt = ES.assign_instances_for_scan(pred, gt_file)

        if visit_id not in scan_cache:
            scan_xyz = L.load_laser_scan_xyz(f'{ROOT}/data', visit_id)
            scan_cache[visit_id] = (scan_xyz, cKDTree(scan_xyz))
        scan_xyz, scan_tree = scan_cache[visit_id]

        pred_motion = {}
        for i, r in enumerate(items):
            n_pred_total += 1
            pts_idx = np.flatnonzero(r['pred_mask'][:n_points])
            if len(pts_idx) < 5:
                continue
            pred_pts = scan_xyz[pts_idx]
            cen = pred_pts.mean(0)
            nbr = wide_neighbourhood(visit_id, scan_xyz, scan_tree, cen)
            pred_motion[i] = MOT.predict_motion(r['label'], pred_pts, nbr)
            n_pred_motion_ok += 1

        gt_motion = {}
        for inst_id, annot_id in inst_to_annot.items():
            gm = gt_motion_for_annot(visit_id, annot_id, scan_xyz)
            gt_motion[inst_id] = gm
            if gm is not None:
                n_gt_motion_ok += 1

        matches_key = os.path.abspath(gt_file)
        for gate in ('raw', 'M', 'MA', 'MAO'):
            g2p = deepcopy(gt2pred); p2g = deepcopy(pred2gt)
            if gate != 'raw':
                for label_name in g2p:
                    for gt in g2p[label_name]:
                        gm = gt_motion.get(gt['instance_id'])
                        gt['matched_pred'] = [p for p in gt['matched_pred']
                                               if motion_correct(gm, pred_motion.get(p['uuid']), gate)]
                for label_name in p2g:
                    for p in p2g[label_name]:
                        pm = pred_motion.get(p['uuid'])
                        p['matched_gt'] = [g for g in p['matched_gt']
                                            if motion_correct(gt_motion.get(g['instance_id']), pm, gate)]
            all_matches[gate][matches_key] = dict(gt=g2p, pred=p2g)

    print(f'GT instances with resolvable motion params: {n_gt_motion_ok}')
    print(f'predicted instances with resolvable motion params: {n_pred_motion_ok}/{n_pred_total}')
    return all_matches

MOTION_MATCHES = build_motion_gated_matches(RESULTS)

print(f"\n{'gate':<8} {'AP25':>8}   meaning")
GATE_RESULTS = {}
for gate, meaning in [('raw', '(sanity check -- must equal Cell 12s AP25 exactly)'),
                       ('M', 'AP25 + correct motion type'),
                       ('MA', '+ axis within 15 deg'),
                       ('MAO', '+ origin MD<0.25m (revolute only)')]:
    ap = ES.evaluate_matches(MOTION_MATCHES[gate])
    avgs = ES.compute_averages(ap)
    GATE_RESULTS[gate] = avgs
    print(f"{gate:<8} {avgs['all_ap_25%']:>7.1f}%   {meaning}")

print(f"\nsanity check: raw AP25 {GATE_RESULTS['raw']['all_ap_25%']:.2f}%  vs  "
      f"Cell 12 AP25 {AVGS['all_ap_25%']:.2f}%  (should match to rounding)")
print(f"\nSceneFun3D's own Mask3D-FM baseline (full supervision, full-scene "
      f"detection, paper Table 5): AP25=26.6%  +M=23.8-24.0%  +MA=9.8-10.2%  +MAO=7.9-8.1%")
print("Not a like-for-like comparison yet -- this run is still under the oracle_label")
print("assumption (Stage 2/3's cleanest condition), not full-scene unprompted detection.")
